# Formulator agent tests

Tests the parser → formulator chain. Run cells top to bottom.

**Setup:** ensure `.env` has `ANTHROPIC_API_KEY` (copy from `.env.example`).

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "orharness").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not set. Copy .env.example to .env and add your key."
    )

print("API key loaded from .env")

API key loaded from .env


In [4]:
from orharness.models import ORHarnessConfig
from orharness.agents.parser import parse_problem
from orharness.agents.formulator import formulate_problem

config = ORHarnessConfig()
print(config)

model='claude-sonnet-4-6' max_retries=3 timeout_seconds=30 temperature=0.0


## Test: scheduling problem (parser → formulator)

First parse the natural-language problem, then formulate it into a math model.
Expect `objective_kind=feasibility` (no minimize/maximize in the input).
For feasibility problems, `objective` should be `null`.

In [5]:
parsed = parse_problem(
    "I have 6 nurses, 3 shifts per day, 7 days a week. "
    "No nurse works more than 5 shifts per week. "
    "Night shifts need at least 2 nurses.",
    config,
)
print("PARSED:")
print(parsed)

PARSED:
is_or_problem=True problem_type=<ProblemType.SCHEDULING: 'scheduling'> confidence=<Confidence.HIGH: 'high'> reason='This is a nurse scheduling problem assigning nurses to shifts across days with coverage and workload constraints.' entities={'nurses': 6, 'shifts_per_day': 3, 'days_per_week': 7, 'max_shifts_per_nurse_per_week': 5, 'min_nurses_night_shift': 2} constraints=['Each nurse works at most 5 shifts per week', 'Night shifts must have at least 2 nurses assigned'] objective='Feasible nurse-to-shift assignment satisfying all constraints, or optionally minimize understaffing or total nurse hours'


In [6]:
formulated = formulate_problem(parsed, config)

print("problem_type:", formulated.problem_type)
print("objective_kind:", formulated.objective_kind)
print("\nobjective:", formulated.objective)

print("\nvariables:")
for v in formulated.variables:
    print("  -", v)

print("\nconstraints:")
for c in formulated.constraints:
    print("  -", c)

print("\nparameters:")
for k, val in formulated.parameters.items():
    print(f"  {k}: {val}")

problem_type: ProblemType.SCHEDULING

objective: Minimize total understaffing on night shifts: sum over k in {0..6} of max(0, 2 - sum over i in {0..5} of x[i][2][k]). Implemented as minimize sum_k s[k] where s[k] = shortage on night shift day k.

variables:
  - x[i][j][k]: binary variable, equals 1 if nurse i is assigned to shift j on day k, where i in {0,1,2,3,4,5}, j in {0,1,2} (0=morning, 1=afternoon, 2=night), k in {0,1,2,3,4,5,6}

constraints:
  - For each nurse i in {0..5}: sum over j in {0,1,2} and k in {0..6} of x[i][j][k] <= 5 (each nurse works at most 5 shifts per week)
  - For each day k in {0..6}: sum over i in {0..5} of x[i][2][k] + s[k] >= 2, where s[k] >= 0 is the night shift shortage on day k (night shifts must have at least 2 nurses, shortage penalized in objective)
  - For each nurse i in {0..5} and each day k in {0..6}: sum over j in {0,1,2} of x[i][j][k] <= 1 (each nurse works at most one shift per day)
  - x[i][j][k] in {0, 1} for all i in {0..5}, j in {0,1,2}, k i

## Test: allocation problem

A different problem type to check the formulator generalizes beyond scheduling.

In [7]:
parsed_alloc = parse_problem(
    "I have a budget of 1000 dollars. There are 5 projects with costs "
    "200, 400, 300, 500, 100 and values 3, 5, 4, 7, 1. "
    "Maximize total value without exceeding the budget.",
    config,
)
print(parsed_alloc)

formulated_alloc = formulate_problem(parsed_alloc, config)
print("\n--- formulated ---")
print(formulated_alloc)

is_or_problem=True problem_type=<ProblemType.ALLOCATION: 'allocation'> confidence=<Confidence.HIGH: 'high'> reason='This is a classic 0/1 knapsack problem where we allocate a fixed budget across projects to maximize total value.' entities={'budget': 1000, 'num_projects': 5, 'costs': [200, 400, 300, 500, 100], 'values': [3, 5, 4, 7, 1]} constraints=['Total cost of selected projects must not exceed 1000 dollars', 'Each project is either fully selected or not selected (binary decision)'] objective='Maximize total value of selected projects'

--- formulated ---
problem_type=<ProblemType.ALLOCATION: 'allocation'> variables=['x[i]: binary variable, 1 if project i is selected, 0 otherwise, for i in {0, 1, 2, 3, 4}'] objective='Maximize sum over i of values[i] * x[i] = 3*x[0] + 5*x[1] + 4*x[2] + 7*x[3] + 1*x[4]' constraints=['Total cost constraint: 200*x[0] + 400*x[1] + 300*x[2] + 500*x[3] + 100*x[4] <= 1000', 'Binary decision: x[i] in {0, 1} for all i in {0, 1, 2, 3, 4}'] parameters={'num_pro

## Test: formulator error handling

These check failure branches in `formulate_problem` directly, by swapping the
LLM call (`completion`) for a fake that returns canned bad output.
No API calls — fast, free, deterministic. Each case must raise `FormulationError`:

1. malformed JSON
2. valid JSON but empty `variables`
3. valid JSON but empty `constraints`
4. feasibility problem but formulator returns an objective (strict validation)

A fifth case confirms valid feasibility output still parses into a `FormulatedModel`.

In [9]:
from types import SimpleNamespace

import orharness.agents.formulator as fmod
from orharness.models import ParsedProblem, ProblemType, Confidence, ObjectiveKind
from orharness.exceptions import FormulationError

# Minimal valid parsed input; its contents don't matter since we fake the LLM reply.
dummy_parsed = ParsedProblem(
    is_or_problem=True,
    problem_type=ProblemType.SCHEDULING,
    confidence=Confidence.HIGH,
    reason="fixture for error-path tests",
    entities={"workers": 4},
    constraints=["each shift needs at least 1 worker"],
    objective_kind=ObjectiveKind.FEASIBILITY,
)


def fake_completion_returning(content: str):
    """Build a stand-in for litellm.completion that always returns `content`."""
    def _fake(*args, **kwargs):
        message = SimpleNamespace(content=content)
        choice = SimpleNamespace(message=message)
        return SimpleNamespace(choices=[choice])
    return _fake


def expect_formulation_error(label: str, fake_content: str):
    fmod.completion = fake_completion_returning(fake_content)
    try:
        fmod.formulate_problem(dummy_parsed, config)
    except FormulationError as e:
        print(f"PASS [{label}] raised FormulationError: {str(e)[:60]}...")
    else:
        print(f"FAIL [{label}] expected FormulationError, none raised")


VALID = (
    '{"problem_type": "scheduling", "objective_kind": "feasibility", '
    '"variables": ["x[i]: binary"], "objective": null, '
    '"constraints": ["c1"], "parameters": {}}'
)

original_completion = fmod.completion
try:
    expect_formulation_error("bad JSON", "this is not json at all")
    expect_formulation_error("empty variables",
        '{"problem_type": "scheduling", "objective_kind": "feasibility", '
        '"variables": [], "objective": null, '
        '"constraints": ["c1"], "parameters": {}}')
    expect_formulation_error("empty constraints",
        '{"problem_type": "scheduling", "objective_kind": "feasibility", '
        '"variables": ["x[i]"], "objective": null, '
        '"constraints": [], "parameters": {}}')
    expect_formulation_error("feasibility with objective",
        '{"problem_type": "scheduling", "objective_kind": "feasibility", '
        '"variables": ["x[i]"], "objective": "minimize cost", '
        '"constraints": ["c1"], "parameters": {}}')

    # Positive control: valid feasibility payload should succeed.
    fmod.completion = fake_completion_returning(VALID)
    result = fmod.formulate_problem(dummy_parsed, config)
    print(f"PASS [valid output] -> {len(result.variables)} var(s), "
          f"{len(result.constraints)} constraint(s), objective={result.objective}")
finally:
    fmod.completion = original_completion  # always restore the real LLM call

PASS [bad JSON] raised FormulationError: Formulator returned invalid JSON: this is not json at all...
PASS [empty variables] raised FormulationError: Formulator returned no decision variables...
PASS [empty constraints] raised FormulationError: Formulator returned no constraints...
PASS [valid output] -> 1 var(s), 1 constraint(s)
